In [76]:
import pandas as pd
import numpy as np
import os
from enum import Enum

# Run-Wise Analysis
In this analysis, we look at the run as a whole. Our goal is to compute single metrics to provide an overall estimate of the models performance. We rely on two metrics:
1. Normalized Root Mean Squared Error (NRMSE)
2. Pearson's Correlation Coefficient (PCC)

### How was the RMSE normalized?
There are different methods for normalization of rmse based on how the data looks. 


We find the error with respect to the maximum signal signal amplitude

| **Normalization**      | **Suitability** | **Notes**                                               |
|------------------------|-----------------|----------------------------------------------------------|
| Mean of `y_true`       | ❌ Poor          | Inflates NRMSE if signal is sparse (our case             |
| Range (`max - min`)    | ⚠️ Okay          | Peaks dominate scaling                                   |
| Standard Deviation     | ⚠️ Mixed         | Can still overweight peaks                               |
| Max of `y_true`        | ✅ Good          | Stable and meaningful for sparse signals                 |
| Segmented NRMSE        | ✅ Advanced      | Best for analyzing different behaviors (e.g. peaks vs. baseline) |

We focus on last two as they apply to our data. Note: this is is stats knowledge which does not have a reference to a paper. How do I cite a reference for it. I found it in blogs mainly https://www.marinedatascience.co/blog/2019/01/07/normalizing-the-rmse/.

⚠️ TODO: do a reference check

#### Can we simply normalize by the max of `y_true`?
The ranges of our 3 axes are different leading to different max values. Therefore, we cannot normalize by this value as this would be kind of cheating for X and Y axes. Therefore, I use the following way to first find rmse for each axis and then take an average to compute a single value (refer to column `Averaged_NRMSE`).

$\text{NRMSE} = \frac{1}{3} \sum_{j \in \{x, y, z\}} \frac{\sqrt{\frac{1}{N} \sum_{i=1}^{N} \left( y_{i,j} - \hat{y}_{i,j} \right)^2}}{\max_i |y_{i,j}|}$

where
- $\mathbf{y}i = [y_{i,x}, y_{i,y}, y_{i,z}]$ is the ground truth
- $\hat{\mathbf{y}}i = [\hat{y}_{i,x}, \hat{y}_{i,y}, \hat{y}_{i,z}]$ is the prediction
- $N$ is the number of samples

In [77]:
losses = ["mse", "huber", "l1"]
results_dir = '../plots' # directory containing the csv files

# Modify the following list for your folder names. The folders are expected to have two CSV files named:
# quantified_results_runwise.csv and quantified_results_peakwise.csv which are generated by the evaluation script.
folder_names = [f'{results_dir}/state-transformer_seq_10_{loss}_epochs_150' for loss in losses]

In [78]:
df_list = pd.DataFrame()

metrics = ['RMSE', 'NRMSE', 'Pearson Correlation']
axes = ['X', 'Y', 'Z']
# Define the columns for the DataFrame using the metrics and axes
cols = metrics + [f'{metric}_{axis}' for metric in metrics for axis in axes] + \
        ['Averaged_NRMSE', 'Averaged_PCC']

for folder_name in folder_names:
    # read run-wise data
    file_path = f'{folder_name}/quantified_results_runwise.csv'
    df_runs = pd.read_csv(file_path)
    # drop duplicate rows
    df_runs = df_runs.drop_duplicates()

    # compute average of the 3 axes NRMSE values
    df_runs['Averaged_NRMSE'] = df_runs[[f'NRMSE_{axis}' for axis in axes]].mean(axis=1)

    # compute average PCC
    df_runs['Averaged_PCC'] = df_runs[[f'Pearson Correlation_{axis}' for axis in axes]].mean(axis=1)

    print(f"Data has info about {df_runs.shape[0]} runs")

    # calculate the mean and std for each metric across all runs
    means = df_runs[cols].mean()
    stds = df_runs[cols].std()

    # Create two DataFrames with renamed columns
    means_df = means.rename(lambda x: f"{x}_mean").to_frame().T
    stds_df = stds.rename(lambda x: f"{x}_std").to_frame().T

    # Interleave columns
    interleaved_columns = []
    for col in cols:
        interleaved_columns.append(f"{col}_mean")
        interleaved_columns.append(f"{col}_std")

    # Combine and reorder
    df_run = pd.concat([means_df, stds_df], axis=1)[interleaved_columns]
   
    # Extract the loss type from the folder name you might need to adjust this based on your folder naming convention
    df_run['loss'] = folder_name.split('_')[-3]  
    df_list = pd.concat([df_list, df_run], ignore_index=True)

# Make loss the index
df_list = df_list.set_index('loss')

Data has info about 42 runs
Data has info about 42 runs
Data has info about 42 runs


In [79]:
df_list[['Averaged_NRMSE_mean', 'Averaged_NRMSE_std', 'NRMSE_mean', 'NRMSE_std', 'RMSE_mean', 'RMSE_std', 'Averaged_PCC_mean', 'Averaged_PCC_std']]

,Averaged_NRMSE_mean,Averaged_NRMSE_std,NRMSE_mean,NRMSE_std,RMSE_mean,RMSE_std,Averaged_PCC_mean,Averaged_PCC_std
loss,,,,,,,,
mse,0.234105,0.195830,0.173419,0.211465,0.215349,0.152247,0.475566,0.284124
huber,0.247819,0.220050,0.195873,0.237592,0.225966,0.144654,0.521829,0.249259
l1,0.223212,0.163298,0.165189,0.175912,0.208874,0.144440,0.503662,0.258386


In [80]:
# for each column, find the loss (index) that has the minimum value for rmse and the maximum value for correlation, print that in a pretty way
for col in df_list.columns:
    if 'RMSE' in col:
        min_loss = df_list[col].idxmin()
        min_value = df_list[col].min()
        print(f"Minimum {col} is {min_value:.4f} for loss {min_loss}")
    elif 'Correlation' in col:
        max_loss = df_list[col].idxmax()
        max_value = df_list[col].max()
        print(f"Maximum {col} is {max_value:.4f} for loss {max_loss}")

Minimum RMSE_mean is 0.2089 for loss l1
Minimum RMSE_std is 0.1444 for loss l1
Minimum NRMSE_mean is 0.1652 for loss l1
Minimum NRMSE_std is 0.1759 for loss l1
Maximum Pearson Correlation_mean is 0.6021 for loss huber
Maximum Pearson Correlation_std is 0.3276 for loss l1
Minimum RMSE_X_mean is 0.1016 for loss l1
Minimum RMSE_X_std is 0.0619 for loss mse
Minimum RMSE_Y_mean is 0.1273 for loss l1
Minimum RMSE_Y_std is 0.0745 for loss mse
Minimum RMSE_Z_mean is 0.3141 for loss l1
Minimum RMSE_Z_std is 0.2414 for loss l1
Minimum NRMSE_X_mean is 0.1939 for loss l1
Minimum NRMSE_X_std is 0.1353 for loss l1
Minimum NRMSE_Y_mean is 0.2345 for loss huber
Minimum NRMSE_Y_std is 0.2196 for loss huber
Minimum NRMSE_Z_mean is 0.2345 for loss l1
Minimum NRMSE_Z_std is 0.2732 for loss l1
Maximum Pearson Correlation_X_mean is 0.3559 for loss huber
Maximum Pearson Correlation_X_std is 0.4450 for loss mse
Maximum Pearson Correlation_Y_mean is 0.5391 for loss huber
Maximum Pearson Correlation_Y_std is 0.

# Peak-by-Peak Analysis

## Temporal Fidelity
A peak might be predicted correctly, late or early. To calculate this, we use *temporal cross-correlation* which works as follows:
1. Consider one peak in the ground truth signal (in case of multiple peaks, we look at them one at a time)
2. Find the correlation of the ground truth signal $f_{gt}(t)$ and the delayed predicted signal $f_{pred}(t+ \tau)$
3. Find the value of $\tau$ which gives the highest value
4. If the value of correlation is high (eg. greater than 0.6), then classify the signal as early, late, correct or false detection.

### Threshold for Classification
We use a threshold $\mathbb{T}$}$ to make the classification. 
1. When $\mathbb{T_1}<|\tau|$ and has a positive sign, the predicted peak is classified as an **late** peak
2. When $\mathbb{T_1}<|\tau|$ and has a negative sign, the predicted peak is classified as an **early** peak
3. When $|\tau|<=\mathbb{T_1}$, the predicted peak is a potential correctly detected peak

The signal frequency is 800 Hz which means that each sample represents 1.25ms. The control frequency for robotic surgery tasks is typically 30 Hz which represents 1/30 seconds. Therefore, we set the threshold $\mathbb{T_1}$ to 25 samples (The max possible value is 26.66667), that means we have an error tolerance of 31.25ms. 

Once the classification is made, we can derive the following quantitative results from this analysis for reporting results:
1. Number/Percentage of Early Detections
2. Number/Percentage of Late Detections
3. Number/Percentage of Correct Detections
4. Number/Percentage of False Detections
5. Average Absolute Time Axis Error: quantitifes the average shift (irrespective of the sign) for the predicted signal. Note that we would need to ignore the false detections when calculating this.

## Structural Fidelity
It is concerned with shape of the signals:
1. Correlation: Is the shape similar? Do they rise and fall at the same time? Therefore, peaks with a correlation below a threshol (0.6) can be labelled as false.
2. Overshoot/Undershoot: We consider different error tolerance (1%, 5% and 10%) for classifying the peak as overshoot/undershoot.

Note: we calculate two kinds of results. One while considering the threshold and other without considering it.



### Helper Functions

In [81]:
# define an enumeration for time axis errors (early, late, false, correct)
class TimeAxisError(Enum):
    """
    Enum for time axis errors.
    """
    EARLY = 0
    LATE = 1
    FALSE = 2
    CORRECT = 3

    @classmethod
    def to_str(cls, error):
        """
        Convert enum to string.
        """
        if error == cls.EARLY:
            return "Early"
        elif error == cls.LATE:
            return "Late"
        elif error == cls.FALSE:
            return "False"
        elif error == cls.CORRECT:
            return "Correct"
        else:
            raise ValueError("Invalid error type")
        
# define an enumeration for force axis errors (overshoot, undershoot, correct)
class ForceAxisError(Enum):
    """
    Enum for force axis errors.
    """
    UNDEFINED = -1
    OVERSHOOT = 0
    UNDERSHOOT = 1
    CORRECT = 2

    @classmethod
    def to_str(cls, error):
        """
        Convert enum to string.
        """
        if error == cls.OVERSHOOT:
            return "Overshoot"
        elif error == cls.UNDERSHOOT:
            return "Undershoot"
        elif error == cls.CORRECT:
            return "Correct"
        else:
            return "Incorrect"

In [ ]:
time_axis_error_key = 'Time-axis Classification'
force_axis_error_key = 'Force-axis Classification'

def classify_peak_time(row, corr_threshold=0.6, lag_threshold=25):
    """Classify the peak based on correlation and lag values.
    
    Parameters:
        row (pd.Series): A row from the DataFrame containing 'Corr' and 'CrossCorr Lag'.
        corr_threshold (float): The correlation threshold for classification.
        lag_threshold (float): The lag threshold for classification. 
    Returns:
        TimeAxisError: The classification of the time axis error.
    """

    corr_key = 'Corr'
    lag_key = 'CrossCorr Lag'
    # Check if the row contains the necessary columns
    if corr_key not in row or lag_key not in row:
        raise ValueError(f"Row must contain {corr_key} and {lag_key} columns.")
    
    T_1 = lag_threshold
    
    if row[corr_key] >= corr_threshold:
        if abs(row[lag_key]) <= T_1:
            return TimeAxisError.CORRECT
        elif row[lag_key] > T_1:
            return TimeAxisError.LATE
        elif row[lag_key] < -T_1:
            return TimeAxisError.EARLY
    
    return TimeAxisError.FALSE

# Magnitude errors: If the peak is correct, early, or late, we can calculate the energy error (normalized difference in energies of the 2 signals) which tells us how far the peak is from the actual peak. Based on that, we can classify it as undershoot, overshoot, or correct.
def classify_magnitude_error(row, threshold=0.01, consider_corr=True): # percentage of height difference
    """Classify the magnitude error based on the peak classification.
    
    Parameters:
        row (pd.Series): A row from the DataFrame containing 'Time-axis Classification' and 'Max Magnitude Difference Norm'.
        threshold (float): The threshold for classifying undershoot and overshoot. This is a percentage of the peak height difference.
    Returns:
        ForceAxisError: The classification of the force axis error.
    """
    
    time_error_key = 'Time-axis Classification'
    energy_error_key = 'Max Magnitude Difference Norm'
    # Check if the row contains the necessary columns
    if time_error_key not in row or energy_error_key not in row:
        raise ValueError(f"Row must contain {time_error_key} and {energy_error_key} columns.")
    
    if consider_corr and row[time_error_key] == TimeAxisError.FALSE:
        return ForceAxisError.UNDEFINED
    else:
        if abs(row[energy_error_key]) < threshold:
            return ForceAxisError.CORRECT
        elif row[energy_error_key] >= threshold:
            return ForceAxisError.OVERSHOOT
        elif row[energy_error_key] <= -threshold:
            return ForceAxisError.UNDERSHOOT

In [ ]:
def aggregate_peakwise_results(file_path, mag_threshold=0.01, corr_threshold=0.6, lag_threshold=25, consider_corr=True):
    """Aggregate the peakwise results from the CSV file.
    Parameters:
        file_path (str): The path to the CSV file containing peakwise results.
    Returns:
        pd.DataFrame: A DataFrame containing the aggregated time axis errors.
        pd.DataFrame: A DataFrame containing the aggregated force axis errors.
    """
    df = pd.read_csv(file_path)
    # drop duplicate rows
    df = df.drop_duplicates()

    print(f"Data has info about {df.Run.unique().shape[0]} runs and {df.shape[0]} peaks")

    # use lambda function to apply the classification functions to the DataFrame
    df[time_axis_error_key] = df.apply(lambda row: classify_peak_time(row, corr_threshold=corr_threshold, lag_threshold=lag_threshold), axis=1)
    df[force_axis_error_key] = df.apply(lambda row: classify_magnitude_error(row, threshold=mag_threshold, consider_corr=consider_corr), axis=1)
    # convert the enum values to strings for better readability
    df[time_axis_error_key] = df[time_axis_error_key].apply(lambda x: TimeAxisError.to_str(x))
    df[force_axis_error_key] = df[force_axis_error_key].apply(lambda x: ForceAxisError.to_str(x))

    # Create a new DataFrame with aggregated results for all runs for each model
    # For the dataframe, we need to count the number of occurences of each time axis error and record the number and percentage of each time axis error
    agg_time_all = df.groupby([time_axis_error_key]).size().reset_index(name='Count')
    agg_time_all['Percentage'] = (agg_time_all['Count'] / agg_time_all['Count'].sum()) * 100
    agg_time_all['Axis'] = 'All'

    agg_time = df.groupby(['Axis', time_axis_error_key]).size().reset_index(name='Count')
    agg_time['Percentage'] = (agg_time['Count'] / agg_time['Count'].sum()) * 100
    agg_time = pd.concat([agg_time, agg_time_all], ignore_index=True)

    agg_force_all = df.groupby([force_axis_error_key]).size().reset_index(name='Count')
    agg_force_all['Percentage'] = (agg_force_all['Count'] / agg_force_all['Count'].sum()) * 100
    agg_force_all['Axis'] = 'All'
    agg_force = df.groupby(['Axis', force_axis_error_key]).size().reset_index(name='Count')
    agg_force['Percentage'] = (agg_force['Count'] / agg_force['Count'].sum()) * 100
    agg_force = pd.concat([agg_force, agg_force_all], ignore_index=True)

    return agg_time, agg_force


### Magnitude Error Threshold of 1%, Considering Correlation value threshold

Change values according to requirements

In [ ]:
df1_list = []
df2_list = []

consider_corr = True  # Set to False if you want to ignore correlation in force axis classification
mag_error_threshold = 0.01  # Percentage of height difference for undershoot/overshoot classification

for folder_name in folder_names:# read peak-wise data
    file_path = f'{folder_name}/quantified_results_peakwise.csv'
    agg_time, agg_force = aggregate_peakwise_results(file_path, 
                                                    mag_threshold=mag_error_threshold, 
                                                    corr_threshold=0.6, 
                                                    lag_threshold=25, 
                                                    consider_corr=consider_corr)
    df1_list.append(agg_time)
    df2_list.append(agg_force)

for i, loss in enumerate(losses):
    df1_list[i]['loss'] = loss
    df2_list[i]['loss'] = loss

df1 = pd.concat(df1_list, ignore_index=True)
df2 = pd.concat(df2_list, ignore_index=True)

Data has info about 42 runs and 153 peaks
Data has info about 42 runs and 153 peaks
Data has info about 42 runs and 153 peaks


In [86]:
df1[df1['Axis'] == 'All'].pivot(index='loss', columns=time_axis_error_key, values='Count').replace(np.nan, 0)

Time-axis Classification,Correct,Early,False,Late
loss,,,,
huber,82,10,52,9
l1,77,8,59,9
mse,71,6,66,10


In [87]:
df1[df1['Axis'] == 'X'].pivot(index='loss', columns=time_axis_error_key, values='Count').replace(np.nan, 0)

Time-axis Classification,Correct,Early,False,Late
loss,,,,
huber,17,4,28,2
l1,15,3,31,2
mse,22,1,26,2


In [88]:
df1[df1['Axis'] == 'Y'].pivot(index='loss', columns=time_axis_error_key, values='Count').replace(np.nan, 0)

Time-axis Classification,Correct,Early,False,Late
loss,,,,
huber,27,4,14,6
l1,25,3,18,5
mse,22,4,19,6


In [89]:
df1[df1['Axis'] == 'Z'].pivot(index='loss', columns=time_axis_error_key, values='Count').replace(np.nan, 0).astype(int)

Time-axis Classification,Correct,Early,False,Late
loss,,,,
huber,38,2,10,1
l1,37,2,10,2
mse,27,1,21,2


In [90]:
df2[df2['Axis'] == 'All'].pivot(index='loss', columns=force_axis_error_key, values='Count').replace(np.nan, 0)

Force-axis Classification,Correct,Incorrect,Overshoot,Undershoot
loss,,,,
huber,2,52,51,48
l1,1,59,41,52
mse,1,66,37,49


In [91]:
df2[df2['Axis'] == 'X'].pivot(index='loss', columns=force_axis_error_key, values='Count').replace(np.nan, 0).astype(int)

Force-axis Classification,Correct,Incorrect,Overshoot,Undershoot
loss,,,,
huber,1,28,9,13
l1,1,31,7,12
mse,0,26,10,15


In [92]:
df2[df2['Axis'] == 'Y'].pivot(index='loss', columns=force_axis_error_key, values='Count').replace(np.nan, 0)

Force-axis Classification,Incorrect,Overshoot,Undershoot
loss,,,
huber,14,18,19
l1,18,13,20
mse,19,11,21


In [93]:
df2[df2['Axis'] == 'Z'].pivot(index='loss', columns=force_axis_error_key, values='Count').replace(np.nan, 0).astype(int)

Force-axis Classification,Correct,Incorrect,Overshoot,Undershoot
loss,,,,
huber,1,10,24,16
l1,0,10,21,20
mse,1,21,16,13
